Baseline

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from narwhals import Categorical
from narwhals.selectors import categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score, recall_score, ConfusionMatrixDisplay, confusion_matrix, roc_curve
from lightgbm.callback import early_stopping, log_evaluation


import lightgbm as lgb

In [5]:
path_to_data = "/home/usl/PycharmProjects/home-credit-default-risk/data/raw/home-credit-default-risk/"
# path_to_data = "/content/drive/MyDrive/Home_Credit_data/"

application_train_df = pd.read_csv(path_to_data+"application_train.csv")
application_test_df = pd.read_csv(path_to_data+"application_test.csv")
sample_submission_df = pd.read_csv(path_to_data+"sample_submission.csv")

### Feature engineering

#### Bureau_df

In [6]:
bureau_df = pd.read_csv(path_to_data+"bureau.csv")
bureau_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1716428 entries, 0 to 1716427
Data columns (total 17 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   SK_ID_CURR              1716428 non-null  int64  
 1   SK_ID_BUREAU            1716428 non-null  int64  
 2   CREDIT_ACTIVE           1716428 non-null  object 
 3   CREDIT_CURRENCY         1716428 non-null  object 
 4   DAYS_CREDIT             1716428 non-null  int64  
 5   CREDIT_DAY_OVERDUE      1716428 non-null  int64  
 6   DAYS_CREDIT_ENDDATE     1610875 non-null  float64
 7   DAYS_ENDDATE_FACT       1082775 non-null  float64
 8   AMT_CREDIT_MAX_OVERDUE  591940 non-null   float64
 9   CNT_CREDIT_PROLONG      1716428 non-null  int64  
 10  AMT_CREDIT_SUM          1716415 non-null  float64
 11  AMT_CREDIT_SUM_DEBT     1458759 non-null  float64
 12  AMT_CREDIT_SUM_LIMIT    1124648 non-null  float64
 13  AMT_CREDIT_SUM_OVERDUE  1716428 non-null  float64
 14  CR

In [7]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
bureau_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau.csv", case = False, na=False)]
display(bureau_description[["Row", "Description"]])

,Row,Description
122,SK_ID_CURR,ID of loan in our sample - one loan in our sam...
123,SK_BUREAU_ID,Recoded ID of previous Credit Bureau credit re...
124,CREDIT_ACTIVE,Status of the Credit Bureau (CB) reported credits
125,CREDIT_CURRENCY,Recoded currency of the Credit Bureau credit
126,DAYS_CREDIT,How many days before current application did c...
127,CREDIT_DAY_OVERDUE,Number of days past due on CB credit at the ti...
128,DAYS_CREDIT_ENDDATE,Remaining duration of CB credit (in days) at t...
129,DAYS_ENDDATE_FACT,Days since CB credit ended at the time of appl...
130,AMT_CREDIT_MAX_OVERDUE,Maximal amount overdue on the Credit Bureau cr...
131,CNT_CREDIT_PROLONG,How many times was the Credit Bureau credit pr...


In [8]:
bureau_credit_count = (bureau_df.groupby("SK_ID_CURR")["SK_ID_BUREAU"].count().rename("BUREAU_CREDIT_COUNT"))
bureau_credit_count.shape

(305811,)

In [9]:
bureau_df["CREDIT_ACTIVE"].value_counts(dropna=False)

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64

In [10]:
bureau_df["IS_ACTIVE_CREDIT"] = (bureau_df["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_active = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_ACTIVE_COUNT = ("IS_ACTIVE_CREDIT", "sum"), BUREAU_ACTIVE_SHARE = ("IS_ACTIVE_CREDIT", "mean")))
bureau_active.head()

,BUREAU_ACTIVE_COUNT,BUREAU_ACTIVE_SHARE
SK_ID_CURR,,
100001,3,0.428571
100002,2,0.250000
100003,1,0.250000
100004,0,0.000000
100005,2,0.666667


In [11]:
bureau_df["DAYS_CREDIT"].describe()

count    1.716428e+06
mean    -1.142108e+03
std      7.951649e+02
min     -2.922000e+03
25%     -1.666000e+03
50%     -9.870000e+02
75%     -4.740000e+02
max      0.000000e+00
Name: DAYS_CREDIT, dtype: float64


max indicates how recently the last loan was taken out,
min indicates how far back the credit history goes,
mean indicates average age of loans

In [12]:
bureau_days_credit = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_LAST_CREDIT_DAYS = ("DAYS_CREDIT", "max"), BUREAU_FIRST_DAYS = ("DAYS_CREDIT", "min"), BUREAU_MEAN_DAYS = ("DAYS_CREDIT", "mean")))
bureau_days_credit.head()

,BUREAU_LAST_CREDIT_DAYS,BUREAU_FIRST_DAYS,BUREAU_MEAN_DAYS
SK_ID_CURR,,,
100001,-49,-1572,-735.000000
100002,-103,-1437,-874.000000
100003,-606,-2586,-1400.750000
100004,-408,-1326,-867.000000
100005,-62,-373,-190.666667



We can obtain the length of the client's credit history:

In [13]:
bureau_days_credit["BUREAU_CREDIT_HISTORY_LENGTH"] = (
    bureau_days_credit["BUREAU_LAST_CREDIT_DAYS"]
    - bureau_days_credit["BUREAU_FIRST_DAYS"])

In [14]:
bureau_days_credit.head()

,BUREAU_LAST_CREDIT_DAYS,BUREAU_FIRST_DAYS,BUREAU_MEAN_DAYS,BUREAU_CREDIT_HISTORY_LENGTH
SK_ID_CURR,,,,
100001,-49,-1572,-735.000000,1523
100002,-103,-1437,-874.000000,1334
100003,-606,-2586,-1400.750000,1980
100004,-408,-1326,-867.000000,918
100005,-62,-373,-190.666667,311


In [15]:
bureau_days_credit.columns

Index(['BUREAU_LAST_CREDIT_DAYS', 'BUREAU_FIRST_DAYS', 'BUREAU_MEAN_DAYS',
       'BUREAU_CREDIT_HISTORY_LENGTH'],
      dtype='object')

In [16]:
bureau_df["CREDIT_DAY_OVERDUE"].describe()

count    1.716428e+06
mean     8.181666e-01
std      3.654443e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.792000e+03
Name: CREDIT_DAY_OVERDUE, dtype: float64

In [17]:
bureau_df["IS_DAY_OVERDUE"] = (bureau_df["CREDIT_DAY_OVERDUE"] > 0).astype(int)

In [18]:
bureau_overdue = (bureau_df.groupby("SK_ID_CURR").agg(BUREAU_MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"), BUREAU_OVERDUE_CREDIT_COUNT=("IS_DAY_OVERDUE", "sum"), BUREAU_OVERDUE_CREDIT_SHARE=("IS_DAY_OVERDUE", "mean")))

bureau_overdue.head(20)

,BUREAU_MAX_DAYS_OVERDUE,BUREAU_OVERDUE_CREDIT_COUNT,BUREAU_OVERDUE_CREDIT_SHARE
SK_ID_CURR,,,
100001,0,0,0.0
100002,0,0,0.0
100003,0,0,0.0
100004,0,0,0.0
100005,0,0,0.0
100007,0,0,0.0
100008,0,0,0.0
100009,0,0,0.0
100010,0,0,0.0


In [19]:
print("Доля записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).mean())
print("Количество записей с просрочкой:", (bureau_df["CREDIT_DAY_OVERDUE"] > 0).sum())

Доля записей с просрочкой: 0.0024568464275809996
Количество записей с просрочкой: 4217


In [20]:
(bureau_overdue["BUREAU_OVERDUE_CREDIT_COUNT"] > 0).mean()

np.float64(0.012635255108547436)

In [21]:
temp = application_train_df[["SK_ID_CURR", "TARGET"]].merge(bureau_overdue, on="SK_ID_CURR", how="left")
temp["HAS_BUREAU_OVERDUE"] = (temp["BUREAU_OVERDUE_CREDIT_COUNT"].fillna(0) > 0).astype(int)
temp.groupby("HAS_BUREAU_OVERDUE")["TARGET"].agg(["count", "mean"])

,count,mean
HAS_BUREAU_OVERDUE,,
0,304114,0.079855
1,3397,0.158964


In [22]:
categorical_cols_bureau_df = ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]
numerical_cols_bureau_df = ["AMT_CREDIT_MAX_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT","AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
special_cols_bureau_df = ["SK_ID_CURR", "SK_ID_BUREAU", "DAYS_CREDIT", "DAYS_CREDIT_ENDDATE","DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE", "CREDIT_DAY_OVERDUE", "CNT_CREDIT_PROLONG"]

In [23]:
print("SPECIAL:", special_cols_bureau_df)
print("\nCATEGORICAL:", categorical_cols_bureau_df)
print("\nNUMERICAL:", numerical_cols_bureau_df)

SPECIAL: ['SK_ID_CURR', 'SK_ID_BUREAU', 'DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'DAYS_CREDIT_UPDATE', 'CREDIT_DAY_OVERDUE', 'CNT_CREDIT_PROLONG']

CATEGORICAL: ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']

NUMERICAL: ['AMT_CREDIT_MAX_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'AMT_ANNUITY']


## On the class

In [ ]:
bureau_df.groupby(["SK_ID_CURR", "CREDIT_ACTIVE"])[["SK_ID_BUREAU"]].count()

In [ ]:
bureau_days_credit_mean = bureau_df.groupby("SK_ID_CURR")[["DAYS_CREDIT"]].mean()
bureau_days_credit_mean

In [ ]:
bureau_days_credit_median = bureau_df.groupby("SK_ID_CURR")[["DAYS_CREDIT"]].median()
bureau_days_credit_median

In [ ]:
bureau_days_credit_mean.merge(bureau_days_credit_median, on="SK_ID_CURR")

#### previous_application_df

In [24]:
previous_application_df = pd.read_csv(path_to_data+"previous_application.csv")

In [25]:
previous_application_df.info(show_counts=True)
previous_application_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  object 
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  object 
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  object 
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-nu

(1670214, 37)

In [26]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
previous_application_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("previous_application.csv", case = False, na=False)]
display(previous_application_description[["Row", "Description"]])

,Row,Description
173,SK_ID_PREV,ID of previous credit in Home credit related t...
174,SK_ID_CURR,ID of loan in our sample
175,NAME_CONTRACT_TYPE,"Contract product type (Cash loan, consumer loa..."
176,AMT_ANNUITY,Annuity of previous application
177,AMT_APPLICATION,For how much credit did client ask on the prev...
178,AMT_CREDIT,Final credit amount on the previous applicatio...
179,AMT_DOWN_PAYMENT,Down payment on the previous application
180,AMT_GOODS_PRICE,Goods price of good that client asked for (if ...
181,WEEKDAY_APPR_PROCESS_START,On which day of the week did the client apply ...
182,HOUR_APPR_PROCESS_START,Approximately at what day hour did the client ...


In [27]:
categorical_cols_prev_app_df = ["NAME_CONTRACT_TYPE", "WEEKDAY_APPR_PROCESS_START","FLAG_LAST_APPL_PER_CONTRACT", "NAME_CASH_LOAN_PURPOSE", "NAME_CONTRACT_STATUS","NAME_PAYMENT_TYPE", "CODE_REJECT_REASON", "NAME_TYPE_SUITE", "NAME_CLIENT_TYPE","NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE", "CHANNEL_TYPE","NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION"]
numerical_cols_prev_app_df = ["AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT","AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED"]
special_cols_prev_app_df = ["SK_ID_PREV", "SK_ID_CURR", "HOUR_APPR_PROCESS_START", "NFLAG_LAST_APPL_IN_DAY", "CNT_PAYMENT", "DAYS_DECISION", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL", "SELLERPLACE_AREA"]

In [28]:
print("SPECIAL:", special_cols_prev_app_df)
print("\nCATEGORICAL:", categorical_cols_prev_app_df)
print("\nNUMERICAL:", numerical_cols_prev_app_df)

SPECIAL: ['SK_ID_PREV', 'SK_ID_CURR', 'HOUR_APPR_PROCESS_START', 'NFLAG_LAST_APPL_IN_DAY', 'CNT_PAYMENT', 'DAYS_DECISION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL', 'SELLERPLACE_AREA']

CATEGORICAL: ['NAME_CONTRACT_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION']

NUMERICAL: ['AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED']


#### bureau_balance_df

In [29]:
bureau_balance_df = pd.read_csv(path_to_data+"bureau_balance.csv")

In [31]:
bureau_balance_df.info(show_counts=True)
bureau_balance_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27299925 entries, 0 to 27299924
Data columns (total 3 columns):
 #   Column          Non-Null Count     Dtype 
---  ------          --------------     ----- 
 0   SK_ID_BUREAU    27299925 non-null  int64 
 1   MONTHS_BALANCE  27299925 non-null  int64 
 2   STATUS          27299925 non-null  object
dtypes: int64(2), object(1)
memory usage: 624.8+ MB


(27299925, 3)

In [32]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
bureau_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("bureau_balance.csv", case = False, na=False)]
display(bureau_balance_description[["Row", "Description"]])

,Row,Description
139,SK_BUREAU_ID,Recoded ID of Credit Bureau credit (unique cod...
140,MONTHS_BALANCE,Month of balance relative to application date ...
141,STATUS,Status of Credit Bureau loan during the month ...


In [33]:
categorical_cols_bureau_balance_df = ["STATUS "]
numerical_cols_bureau_balance_df = []
special_cols_bureau_balance_df = ["SK_ID_BUREAU", "MONTHS_BALANCE"]

In [34]:
print("SPECIAL:", special_cols_bureau_balance_df)
print("\nCATEGORICAL:", categorical_cols_bureau_balance_df)
print("\nNUMERICAL:", numerical_cols_bureau_balance_df)

SPECIAL: ['SK_ID_BUREAU', 'MONTHS_BALANCE']

CATEGORICAL: ['STATUS ']

NUMERICAL: []


#### POS_CASH_balance_df

In [35]:
POS_CASH_balance_df = pd.read_csv(path_to_data+"POS_CASH_balance.csv")

In [36]:
POS_CASH_balance_df.info(show_counts=True)
POS_CASH_balance_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Non-Null Count     Dtype  
---  ------                 --------------     -----  
 0   SK_ID_PREV             10001358 non-null  int64  
 1   SK_ID_CURR             10001358 non-null  int64  
 2   MONTHS_BALANCE         10001358 non-null  int64  
 3   CNT_INSTALMENT         9975287 non-null   float64
 4   CNT_INSTALMENT_FUTURE  9975271 non-null   float64
 5   NAME_CONTRACT_STATUS   10001358 non-null  object 
 6   SK_DPD                 10001358 non-null  int64  
 7   SK_DPD_DEF             10001358 non-null  int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 610.4+ MB


(10001358, 8)

In [37]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
POS_CASH_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("POS_CASH_balance.csv", case = False, na=False)]
display(POS_CASH_balance_description[["Row", "Description"]])


,Row,Description
142,SK_ID_PREV,ID of previous credit in Home Credit related t...
143,SK_ID_CURR,ID of loan in our sample
144,MONTHS_BALANCE,Month of balance relative to application date ...
145,CNT_INSTALMENT,Term of previous credit (can change over time)
146,CNT_INSTALMENT_FUTURE,Installments left to pay on the previous credit
147,NAME_CONTRACT_STATUS,Contract status during the month
148,SK_DPD,DPD (days past due) during the month of previo...
149,SK_DPD_DEF,DPD during the month with tolerance (debts wit...


In [38]:
categorical_cols_POS_CASH_bal_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_POS_CASH_bal_df = ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE"]
special_cols_POS_CASH_bal_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [39]:
print("SPECIAL:", special_cols_POS_CASH_bal_df)
print("\nCATEGORICAL:", categorical_cols_POS_CASH_bal_df)
print("\nNUMERICAL:", numerical_cols_POS_CASH_bal_df)

SPECIAL: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'SK_DPD', 'SK_DPD_DEF']

CATEGORICAL: ['NAME_CONTRACT_STATUS']

NUMERICAL: ['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE']


#### installments_payments_df

In [40]:
installments_payments_df = pd.read_csv(path_to_data+"installments_payments.csv")

In [41]:
installments_payments_df.info(show_counts=True)
installments_payments_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Non-Null Count     Dtype  
---  ------                  --------------     -----  
 0   SK_ID_PREV              13605401 non-null  int64  
 1   SK_ID_CURR              13605401 non-null  int64  
 2   NUM_INSTALMENT_VERSION  13605401 non-null  float64
 3   NUM_INSTALMENT_NUMBER   13605401 non-null  int64  
 4   DAYS_INSTALMENT         13605401 non-null  float64
 5   DAYS_ENTRY_PAYMENT      13602496 non-null  float64
 6   AMT_INSTALMENT          13605401 non-null  float64
 7   AMT_PAYMENT             13602496 non-null  float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


(13605401, 8)

In [42]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
inst_payment_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("POS_CASH_balance.csv", case = False, na=False)]
display(inst_payment_description[["Row", "Description"]])

,Row,Description
142,SK_ID_PREV,ID of previous credit in Home Credit related t...
143,SK_ID_CURR,ID of loan in our sample
144,MONTHS_BALANCE,Month of balance relative to application date ...
145,CNT_INSTALMENT,Term of previous credit (can change over time)
146,CNT_INSTALMENT_FUTURE,Installments left to pay on the previous credit
147,NAME_CONTRACT_STATUS,Contract status during the month
148,SK_DPD,DPD (days past due) during the month of previo...
149,SK_DPD_DEF,DPD during the month with tolerance (debts wit...


In [44]:
categorical_cols_inst_payment_df = []
numerical_cols_inst_payment_df = ["AMT_INSTALMENT", "AMT_PAYMENT"]
special_cols_inst_payment_df = ["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"]

In [45]:
print("SPECIAL:", special_cols_inst_payment_df)
print("\nCATEGORICAL:", categorical_cols_inst_payment_df)
print("\nNUMERICAL:", numerical_cols_inst_payment_df)

SPECIAL: ['SK_ID_PREV', 'SK_ID_CURR', 'NUM_INSTALMENT_VERSION', 'NUM_INSTALMENT_NUMBER', 'DAYS_INSTALMENT', 'DAYS_ENTRY_PAYMENT']

CATEGORICAL: []

NUMERICAL: ['AMT_INSTALMENT', 'AMT_PAYMENT']


#### credit_card_balance_df

In [47]:
credit_card_balance_df = pd.read_csv(path_to_data+"credit_card_balance.csv")

In [48]:
credit_card_balance_df.info(show_counts=True)
credit_card_balance_df.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   SK_ID_PREV                  3840312 non-null  int64  
 1   SK_ID_CURR                  3840312 non-null  int64  
 2   MONTHS_BALANCE              3840312 non-null  int64  
 3   AMT_BALANCE                 3840312 non-null  float64
 4   AMT_CREDIT_LIMIT_ACTUAL     3840312 non-null  int64  
 5   AMT_DRAWINGS_ATM_CURRENT    3090496 non-null  float64
 6   AMT_DRAWINGS_CURRENT        3840312 non-null  float64
 7   AMT_DRAWINGS_OTHER_CURRENT  3090496 non-null  float64
 8   AMT_DRAWINGS_POS_CURRENT    3090496 non-null  float64
 9   AMT_INST_MIN_REGULARITY     3535076 non-null  float64
 10  AMT_PAYMENT_CURRENT         3072324 non-null  float64
 11  AMT_PAYMENT_TOTAL_CURRENT   3840312 non-null  float64
 12  AMT_RECEIVABLE_PRINCIPAL    3840312 non-null  float64
 1

(3840312, 23)

In [49]:
home_credit_columns_description_df = pd.read_csv(path_to_data+"HomeCredit_columns_description.csv", encoding="cp1252")
credit_card_balance_description = home_credit_columns_description_df[home_credit_columns_description_df["Table"].astype(str).str.contains("credit_card_balance.csv", case = False, na=False)]
display(credit_card_balance_description[["Row", "Description"]])

,Row,Description
150,SK_ID_PREV,ID of previous credit in Home credit related t...
151,SK_ID_CURR,ID of loan in our sample
152,MONTHS_BALANCE,Month of balance relative to application date ...
153,AMT_BALANCE,Balance during the month of previous credit
154,AMT_CREDIT_LIMIT_ACTUAL,Credit card limit during the month of the prev...
155,AMT_DRAWINGS_ATM_CURRENT,Amount drawing at ATM during the month of the ...
156,AMT_DRAWINGS_CURRENT,Amount drawing during the month of the previou...
157,AMT_DRAWINGS_OTHER_CURRENT,Amount of other drawings during the month of t...
158,AMT_DRAWINGS_POS_CURRENT,Amount drawing or buying goods during the mont...
159,AMT_INST_MIN_REGULARITY,Minimal installment for this month of the prev...


In [50]:
categorical_cols_credit_card_balance_df = ["NAME_CONTRACT_STATUS"]
numerical_cols_credit_card_balance_df = ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE", "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT", "CNT_DRAWINGS_OTHER_CURRENT", "CNT_DRAWINGS_POS_CURRENT", "CNT_INSTALMENT_MATURE_CUM"]
special_cols_credit_card_balance_df = ["SK_ID_PREV", "SK_ID_CURR", "MONTHS_BALANCE", "SK_DPD", "SK_DPD_DEF"]

In [51]:
print("SPECIAL:", special_cols_credit_card_balance_df)
print("\nCATEGORICAL:", categorical_cols_credit_card_balance_df)
print("\nNUMERICAL:", numerical_cols_credit_card_balance_df)

SPECIAL: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'SK_DPD', 'SK_DPD_DEF']

CATEGORICAL: ['NAME_CONTRACT_STATUS']

NUMERICAL: ['AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM']


Split into train and val

In [ ]:
train_users = application_train_df["SK_ID_CURR"].unique()
train_ids, val_ids = train_test_split(train_users, test_size=0.2, random_state=42)
train_ids

In [ ]:
train_df = application_train_df[application_train_df.SK_ID_CURR.isin(train_ids)].copy()
val_df = application_train_df[application_train_df.SK_ID_CURR.isin(val_ids)].copy()

In [ ]:
train_df.shape, val_df.shape

In [ ]:
application_train_df["TARGET"].mean(), train_df["TARGET"].mean(), val_df["TARGET"].mean()

In [ ]:
drop_feat = ["SK_ID_CURR"]
train_df.drop(drop_feat, axis=1, inplace=True)
val_df.drop(drop_feat, axis=1, inplace=True)
application_test_df.drop(drop_feat, axis=1, inplace=True)

In [ ]:
X_train = train_df.drop("TARGET", axis=1)
y_train = train_df["TARGET"]
X_val = val_df.drop("TARGET", axis=1)
y_val = val_df["TARGET"]

In [ ]:
cat_cols = X_train.select_dtypes(include = 'object').columns.tolist()
len(cat_cols)

In [ ]:
for col in cat_cols:
    categories = X_train[col].dropna().unique()
    X_train[col] = pd.Categorical(X_train[col], categories = categories)
    X_val[col] = pd.Categorical(X_val[col], categories = categories)


In [ ]:
X_train.select_dtypes(include='object').columns

### LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(n_estimators=1000, max_depth=4, learning_rate=0.1, random_state = 42, n_jobs=-1)
lgb_clf.fit(X_train, y_train, eval_set = [(X_val, y_val)], eval_metric='auc', categorical_feature=cat_cols, callbacks = [early_stopping(stopping_rounds=100), log_evaluation(period = 100)])

threshold = 0.5
val_pred_proba = lgb_clf.predict_proba(X_val)[:, 1]
val_pred = (val_pred_proba >= threshold).astype(int)


roc_auc_lgb = roc_auc_score(y_val, val_pred_proba)
prec_lgb = precision_score(y_val, val_pred, zero_division=0)
rec_lgb = recall_score(y_val, val_pred, zero_division=0)

print("Best_iteration:", lgb_clf.best_iteration_)
print("ROC-AUC_lgb:", roc_auc_lgb)
print("Precision_lgb:", prec_lgb)
print("Recall_lgb:", rec_lgb)

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, val_pred)
plt.title("Confusion Matrix LightGBM")

And now we will calculate the roc_curve and plot a graph:

In [ ]:
fpr, tpr, thresholds = roc_curve(y_val, val_pred_proba)
plt.figure(figsize=(7, 6))

plt.plot(fpr, tpr, label = f"LightGBM ROC-AUC={roc_auc_lgb:.4f}")
plt.plot([0, 1], [0, 1], 'k--', label = "Random classifier ROC-AUC=0.5")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve LightGBM")
plt.legend(loc="lower right")
plt.grid()

We will prepare the test part for training:

In [ ]:
application_test_df.shape

In [ ]:
X_test = application_test_df

In [ ]:
for col in cat_cols:
    X_test[col] = pd.Categorical(X_test[col], categories = X_train[col].cat.categories)
X_test.shape, X_train.shape

In [ ]:
list(X_train.columns) == list(X_test.columns)

In [ ]:
test_pred_proba = lgb_clf.predict_proba(X_test)[:, 1]

In [ ]:
submission = sample_submission_df.copy()
submission["TARGET"] = test_pred_proba
submission.head()

In [ ]:
submission.shape, application_test_df.shape

In [ ]:
submission.to_csv("submission_lgb_baseline.csv", index=False)